# Orientation: Explore Your Routing Infrastructure

This notebook helps you explore what's already been set up in your Snowflake account for the **Fleet Intelligence with Cortex Code** quickstart.

Run each cell to inspect the pre-provisioned infrastructure — image repositories, container images, compute pools, warehouses, and services.

In [ ]:
%%sql -r session_context
SELECT CURRENT_ROLE() AS role,
       CURRENT_WAREHOUSE() AS warehouse,
       CURRENT_USER() AS username,
       CURRENT_ACCOUNT_NAME() AS account

## Database & Schemas
The **OPENROUTESERVICE_APP** database holds all routing infrastructure.

In [ ]:
%%sql -r ors_schemas
SHOW SCHEMAS IN DATABASE OPENROUTESERVICE_APP

## Image Repository
Container images are stored in the **CORE.IMAGE_REPOSITORY**.

| Image | Purpose |
|---|---|
| `openrouteservice` | The core ORS routing engine |
| `ors_control_app` | React + Express dashboard |
| `routing_reverse_proxy` | Nginx proxy |
| `vroom-docker` | VROOM solver |
| `downloader` | Downloads OSM map files |

In [ ]:
%%sql -r image_repos
SHOW IMAGE REPOSITORIES IN DATABASE OPENROUTESERVICE_APP

In [ ]:
%%sql -r images
SHOW IMAGES IN IMAGE REPOSITORY OPENROUTESERVICE_APP.CORE.IMAGE_REPOSITORY

## Compute Pools
Compute pools provide the VM nodes that run containerised services (SPCS).

In [ ]:
%%sql -r compute_pools
SHOW COMPUTE POOLS

## Available Instance Families

In [ ]:
%%sql -r instance_families
SHOW COMPUTE POOL INSTANCE FAMILIES

## Warehouses

In [ ]:
%%sql -r warehouses
SHOW WAREHOUSES

## Services (SPCS)
No services are deployed yet. To deploy, ask Cortex Code: **Build the routing solution**

In [ ]:
%%sql -r services
SHOW SERVICES IN DATABASE OPENROUTESERVICE_APP

## Stages

In [ ]:
%%sql -r stages
SHOW STAGES IN DATABASE OPENROUTESERVICE_APP

---
## What's Next?

Close this notebook and use **Cortex Code** to deploy and run demos.

### Deploy Everything (Recommended)

Paste this single prompt to deploy the full routing solution and all demos in the correct order:

> **Prompt:**
>
> ```
> Deploy the full fleet intelligence stack in order:
> 1. Build the routing solution (compute pools, SPCS services, stages, map files, routing SQL functions)
> 2. Run the fleet intelligence taxi demo (GPS telemetry + React dashboard)
> 3. Setup food delivery fleet intelligence (courier telemetry + React app)
> 4. Run the route optimization demo (VRP with VROOM solver)
> 5. Deploy the retail catchment demo (isochrone trade area analysis)
> 6. Deploy route deviation analysis (detour detection ETL + React dashboard)
> 7. Deploy dwell analysis (Dynamic Table pipeline)
> 8. Setup the routing agent (Snowflake Intelligence agent)
> 9. Deploy the Fleet Explorer Streamlit app (pydeck maps, directions, isochrones, VRP, H3 matrix)
> 10. Setup Agent Playground
> ```

---

### Or Deploy Individually

**Industry Scenarios:**

| Prompt | Industry | What It Does |
|---|---|---|
| `Run the fleet intelligence taxi demo` | Transport & Logistics | Taxi GPS telemetry generation + React fleet dashboard |
| `Setup food delivery fleet intelligence` | Food & Delivery | Courier telemetry with restaurant/delivery simulation |
| `Run the route optimization demo` | Supply Chain | Vehicle Routing Problem (VRP) with VROOM solver |
| `Deploy the retail catchment demo` | Retail & Real Estate | Isochrone-based trade area + competitor analysis |
| `Deploy route deviation analysis` | Fleet Compliance | Detour detection ETL + deviation dashboard |
| `Deploy dwell analysis` | Warehousing & Distribution | 12-step Dynamic Table pipeline (dwell, congestion, SLA) |

**Platform Capabilities:**

| Prompt | What It Does |
|---|---|
| `Build the routing solution` | Creates compute pools, SPCS services, stages, map files, and routing SQL functions |
| `Setup the routing agent` | Snowflake Intelligence agent wrapping ORS for natural language route planning |
| `Deploy the Fleet Explorer Streamlit app` | Interactive pydeck maps with directions, isochrones, VRP, and H3 matrix |

### Configuration

| Prompt | What It Does |
|---|---|
| `Change location to London` | Switches the ORS map region |
| `Change routing profile to cycling` | Adjusts ORS for different travel modes |

In [ ]:
%%sql -r fact_trips_preview
SELECT * FROM SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS LIMIT 10

In [ ]:
%%sql -r table_row_counts
SELECT 'FACT_TRIPS' AS table_name, COUNT(*) AS row_count FROM SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS
UNION ALL
SELECT 'FACT_VEHICLE_TELEMETRY', COUNT(*) FROM SYNTHETIC_DATASETS.UNIFIED.FACT_VEHICLE_TELEMETRY
UNION ALL
SELECT 'DIM_FLEET', COUNT(*) FROM SYNTHETIC_DATASETS.UNIFIED.DIM_FLEET
UNION ALL
SELECT 'DIM_POIS', COUNT(*) FROM SYNTHETIC_DATASETS.UNIFIED.DIM_POIS

In [ ]:
%%sql -r trip_stats
SELECT
  AVG(DURATION_MINUTES) AS avg_duration_min,
  MIN(DURATION_MINUTES) AS min_duration_min,
  MAX(DURATION_MINUTES) AS max_duration_min,
  STDDEV(DURATION_MINUTES) AS stddev_duration_min,
  AVG(DISTANCE_KM) AS avg_distance_km,
  MIN(DISTANCE_KM) AS min_distance_km,
  MAX(DISTANCE_KM) AS max_distance_km,
  STDDEV(DISTANCE_KM) AS stddev_distance_km
FROM SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS

In [ ]:
%%sql -r busiest_vehicles
SELECT f.VEHICLE_ID, d.VEHICLE_TYPE, COUNT(*) AS trip_count
FROM SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS f
JOIN SYNTHETIC_DATASETS.UNIFIED.DIM_FLEET d ON f.VEHICLE_ID = d.VEHICLE_ID
GROUP BY f.VEHICLE_ID, d.VEHICLE_TYPE
ORDER BY trip_count DESC
LIMIT 10

In [ ]:
%%sql -r trips_by_dow
SELECT DAYNAME(TO_TIMESTAMP(TRIP_START)) AS day_of_week,
       DAYOFWEEK(TO_TIMESTAMP(TRIP_START)) AS day_num,
       COUNT(*) AS trip_count
FROM SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS
GROUP BY day_of_week, day_num
ORDER BY day_num

In [ ]:
%%sql -r poi_categories
SELECT CATEGORY, COUNT(*) AS poi_count
FROM SYNTHETIC_DATASETS.UNIFIED.DIM_POIS
GROUP BY CATEGORY
ORDER BY poi_count DESC
LIMIT 10

In [ ]:
%%sql -r trips_per_hour
SELECT HOUR(TO_TIMESTAMP(TRIP_START)) AS hour_of_day,
       COUNT(*) AS trip_count
FROM SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS
GROUP BY hour_of_day
ORDER BY hour_of_day

In [ ]:
%%sql -r cumulative_distance
SELECT DATE_TRUNC('hour', TO_TIMESTAMP(TRIP_START)) AS trip_hour,
       SUM(DISTANCE_KM) AS hourly_distance_km,
       SUM(SUM(DISTANCE_KM)) OVER (ORDER BY trip_hour) AS cumulative_distance_km
FROM SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS
GROUP BY trip_hour
ORDER BY trip_hour

---
## Exercises: Explore the Seed Data

The pre-loaded seed data simulates a fleet of **50 e-bike couriers** operating across San Francisco over 7 days.

Copy the prompt below into the **Cortex Code chat** and it will generate all the exercise cells at once:

---

> **Prompt:**
>
> ```
> Create the following SQL cells in this notebook (one cell per query):
>
> 1. Show the first 10 rows of SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS
> 2. Count the total rows in each table: FACT_TRIPS, FACT_VEHICLE_TELEMETRY, DIM_FLEET, and DIM_POIS in SYNTHETIC_DATASETS.UNIFIED
> 3. Calculate the average, min, max and standard deviation of DURATION_MINUTES and DISTANCE_KM from SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS
> 4. Show the top 10 busiest vehicles by trip count from SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS joined with DIM_FLEET
> 5. Count trips by day of week from SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS
> 6. Show the top 10 POI categories by count from SYNTHETIC_DATASETS.UNIFIED.DIM_POIS
> 7. Count trips per hour of day from SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS
> 8. Show cumulative distance covered by the fleet over time (hourly) from SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS
> ```

---

### Bonus: Deploy the Fleet Explorer App

This deploys a **Streamlit app** with interactive maps, routing directions, and isochrone visualisations. It checks ORS service health before enabling routing features.

> **Prompt:**
>
> ```
> Create and deploy a Streamlit app in this workspace called "fleet-map" on warehouse DEFAULT_WH.
> Add pydeck to environment.yml (snowflake conda channel). Use get_active_session() for the connection.
> The app should have 3 tabs:
>
> Tab 1 - POI Map: Plot all POIs from SYNTHETIC_DATASETS.UNIFIED.DIM_POIS (columns: NAME, CATEGORY, LNG, LAT)
>   on a pydeck ScatterplotLayer with dark map style, with a category filter multiselect.
>
> Tab 2 - Directions: Let users pick 2 POIs from DIM_POIS, then call
>   OPENROUTESERVICE_APP.CORE.DIRECTIONS('driving-car', ARRAY_CONSTRUCT(lng1,lat1), ARRAY_CONSTRUCT(lng2,lat2))
>   and show the route on a GeoJsonLayer with distance/duration metrics.
>
> Tab 3 - Isochrones: Let users pick a POI and a travel time slider (1-15 min), then call
>   OPENROUTESERVICE_APP.CORE.ISOCHRONES('driving-car', lng::FLOAT, lat::FLOAT, minutes::NUMBER)
>   and show the reachability polygon on a GeoJsonLayer.
>
> Add a sidebar health check that calls OPENROUTESERVICE_APP.CORE.CHECK_HEALTH()
> Deploy the app to SYNTHETIC_DATASETS.UNIFIED.FLEET_MAP.
> ```

---

## ORS Control App (SPCS React Dashboard)

Now that you've experienced the core ORS routing functions (directions, isochrones, matrix, optimisation) through SQL and Streamlit, let's see the full potential.

The **ORS Control App** is a production-grade React + Express dashboard running on Snowpark Container Services. It provides:

| Feature | Description |
|---|---|
| **Region Management** | Add, remove, and monitor ORS map regions (cities) |
| **Service Health** | Real-time status of all SPCS services (ORS engine, gateway, VROOM, downloader) |
| **Interactive Routing** | Point-and-click directions, isochrones, and matrix on a full deck.gl map |
| **Fleet Monitoring** | Live vehicle positions, route replay, and telemetry overlays |
| **Configuration** | Change routing profiles, adjust engine settings, manage compute pools |

This app demonstrates what's possible when you combine ORS + SPCS + a modern frontend framework.

Run the cell below to get the endpoint URL:

In [ ]:
%%sql -r dataframe_1


In [ ]:
%%sql -r control_app_endpoint
SHOW ENDPOINTS IN SERVICE OPENROUTESERVICE_APP.CORE.ORS_CONTROL_APP;
SELECT 'https://' || "ingress_url" AS control_app_url FROM TABLE(RESULT_SCAN(LAST_QUERY_ID())) WHERE "name" = 'ors-control-app'